# Kepler Exoplanet Candidate Classification

# 0: Loading and inspecting the Dataset

In this part, we load the NASA Kepler Objects of Interest (KOI) Cumulative Table that we downloaded and put in the "data" subfolder.
We inspect the dataset's dimensions and display the first 3 rows to see what features we have.

In [51]:
import pandas as pd
import numpy as np

# Load the dataset from the data folder
df_raw = pd.read_csv('data/cumulative.csv', comment='#')

#dimensions (Rows and Columns)
print(f"Dataset Shape: {df_raw.shape}")

#Display first 3 rows
display(df_raw.head(3))

Dataset Shape: (9564, 140)


,kepid,kepoi_name,kepler_name,koi_disposition,koi_vet_stat,koi_vet_date,koi_pdisposition,koi_score,koi_fpflag_nt,koi_fpflag_ss,...,koi_dicco_mdec,koi_dicco_mdec_err,koi_dicco_msky,koi_dicco_msky_err,koi_dikco_mra,koi_dikco_mra_err,koi_dikco_mdec,koi_dikco_mdec_err,koi_dikco_msky,koi_dikco_msky_err
0,10797460,K00752.01,Kepler-227 b,CONFIRMED,Done,2018-08-16,CANDIDATE,1.000,0,0,...,0.200,0.16,0.200,0.170,0.080,0.130,0.310,0.170,0.320,0.160
1,10797460,K00752.02,Kepler-227 c,CONFIRMED,Done,2018-08-16,CANDIDATE,0.969,0,0,...,0.000,0.48,0.390,0.360,0.490,0.340,0.120,0.730,0.500,0.450
2,10811496,K00753.01,NaN,CANDIDATE,Done,2018-08-16,CANDIDATE,0.000,0,0,...,-0.034,0.07,0.042,0.072,0.002,0.071,-0.027,0.074,0.027,0.074


In [52]:
#Counting how many rows belong to each category
print("Original Target Distribution:")
print(df_raw['koi_disposition'].value_counts(dropna=False))

print("\nUnique stars (kepid):", df_raw["kepid"].nunique(), "of", len(df_raw), "rows")
print("Duplicate KOI names:", df_raw["kepoi_name"].duplicated().sum())

#Grouping the data by each individual star
stars = df_raw.groupby("kepid").size()
print("KOIs per star:")
#counts how many candidate objects/planets belong to that specific star
print(stars.value_counts().sort_index())

Original Target Distribution:
koi_disposition
FALSE POSITIVE    4839
CONFIRMED         2748
CANDIDATE         1977
Name: count, dtype: int64

Unique stars (kepid): 8214 of 9564 rows
Duplicate KOI names: 0
KOIs per star:
1    7276
2     661
3     179
4      68
5      25
6       3
7       2
Name: count, dtype: int64


Checking data types and missing values

In [53]:
num_cols= df_raw.select_dtypes(include=np.number).columns
obj_cols= df_raw.select_dtypes(exclude=np.number).columns

print(f"There is {len(num_cols)} numeric columns, and {len(obj_cols)} non-numeric columns")
print("Non-numeric:", list(obj_cols))

#calculate the percentage of missing values per col:
missing= (df_raw.isna().mean()*100).sort_values(ascending=False)
print(missing)


There is 123 numeric columns, and 17 non-numeric columns
Non-numeric: ['kepoi_name', 'kepler_name', 'koi_disposition', 'koi_vet_stat', 'koi_vet_date', 'koi_pdisposition', 'koi_disp_prov', 'koi_comment', 'koi_fittype', 'koi_limbdark_mod', 'koi_parm_prov', 'koi_tce_delivname', 'koi_quarters', 'koi_trans_mod', 'koi_datalink_dvr', 'koi_datalink_dvs', 'koi_sparprov']
koi_sma_err2      100.0
koi_sma_err1      100.0
koi_teq_err1      100.0
koi_teq_err2      100.0
koi_longp_err2    100.0
                  ...  
koi_time0bk         0.0
koi_time0           0.0
koi_disp_prov       0.0
dec                 0.0
ra                  0.0
Length: 140, dtype: float64


# 1:Data Leakage removal & target definition

1. **Leaky Feature Elimination:**
   - **koi_score and koi_pdisposition:** these are scored calculated after the final verdict is determined. If the model sees them, it is like they are seing the answer
   - **kepid, kepoi_name:** These are identifiers
   - **koi_fpflag_nt, koi_fpflag_ss, koi_fpflag_co, koi_fpflag_ec:** They are the exact diagnostic tests scientists use to categorize a false positive. If we leave them, it means that the model is copying human decisions instead of learning patterns
   - **koi_vet_stat, koi_vet_date, koi_disp_prov, koi_commen koi_tce_delivname:** These are human administrative notes and timestamps. They don't contain any physical properties about the planet/star
   - **kepler_name:** Only confirmed planets have names in this column. So if a row has a "kepler_name", the model knows immediately that it belongs to class 1

2. **Target Definition (`koi_disposition`):** 
   - CONFIRMED =1 (it is a planet)
     FALSE POSIIVE =0 (not a planet)
   - In the dataset there is a third category which is "CANDIDATE" which means that object haven't been fully verified by NASA, we don't know which class it belongs to. So we are going to exclude it from training and testing
   
3. **Data Filtering:**
   - We automatically drop features that are 100% missing (`NaN`) or contain zero variance (constant values across all rows), because they don't carry any useful predictive info





In [54]:
LABEL = "koi_disposition"

LEAKY= ["koi_pdisposition", "koi_score", "kepler_name",
    "koi_fpflag_nt", "koi_fpflag_ss", "koi_fpflag_co", "koi_fpflag_ec",
    "koi_vet_stat", "koi_vet_date", "koi_disp_prov", "koi_comment",
    "koi_tce_delivname"]

IDS= ["kepid", "kepoi_name", "rowid"]


In [55]:
df_candidate = df_raw[df_raw[LABEL] == "CANDIDATE"].copy()
df = df_raw[df_raw[LABEL].isin(["CONFIRMED", "FALSE POSITIVE"])].copy()
df["y"]= (df[LABEL] == "CONFIRMED").astype(int)
groups= df["kepid"].values

print("Labeled rows:", len(df), "\nCANDIDATE rows:", len(df_candidate))
print()
print(df["y"].value_counts().rename({1: "CONFIRMED", 0: "FALSE POSITIVE"}))
print(f"Positive rate: {df['y'].mean():.3f}")

Labeled rows: 7587 
CANDIDATE rows: 1977

y
FALSE POSITIVE    4839
CONFIRMED         2748
Name: count, dtype: int64
Positive rate: 0.362


In [56]:
#drop_always contains the IDs and the target labels + LEAKY features
drop_always = set(IDS + [LABEL, "y"]+ LEAKY)
#numeric values except IDs and target labels
numeric_all = []
for col in df.select_dtypes(include="number").columns:
  if col not in drop_always:
    numeric_all.append(col)

# remove the columns whose values are all missing(100% empty)
all_nan = []
clean_features = []
for col in numeric_all:
  if df[col].isna().all():
    all_nan.append(col)
  else:
    clean_features.append(col)

# remove the columns who have only 1 value ()
const = []
final_features = []
for col in clean_features:
  if df[col].nunique(dropna=True) <= 1:
    const.append(col)
  else:
    final_features.append(col)
clean_features = final_features

# Verify that koi_time0 is just a shifted offset of koi_time0bk, and errors match
print(np.allclose(df["koi_time0"] - df["koi_time0bk"], 2454833.0))
for s in ["_err1", "_err2"]:
  print(s, np.allclose(df["koi_time0" + s], df["koi_time0bk" + s], equal_nan=True))

# Drop redundant time columns, keeping the 'bk' versions
redundant_time_cols = ["koi_time0", "koi_time0_err1", "koi_time0_err2"]
clean_features = [c for c in clean_features if c not in redundant_time_cols]

print(f"\nClean features remaining after dropping redundant time cols: {len(clean_features)}")

print(f"Clean features remaining: {len(clean_features)}")
print("Dropped all-NaN columns:", all_nan)
print("Dropped constant columns:", const)

True
_err1 True
_err2 True

Clean features remaining after dropping redundant time cols: 92
Clean features remaining: 92
Dropped all-NaN columns: ['koi_eccen_err1', 'koi_eccen_err2', 'koi_longp', 'koi_longp_err1', 'koi_longp_err2', 'koi_ingress', 'koi_ingress_err1', 'koi_ingress_err2', 'koi_sma_err1', 'koi_sma_err2', 'koi_incl_err1', 'koi_incl_err2', 'koi_teq_err1', 'koi_teq_err2', 'koi_model_dof', 'koi_model_chisq', 'koi_sage', 'koi_sage_err1', 'koi_sage_err2']
Dropped constant columns: ['koi_eccen', 'koi_ldm_coeff4', 'koi_ldm_coeff3']


# 2 Train-Test Split


Multiple KOIs can orbit the exact same star and share the same star characteristics. So we need to group our split by kepid to ensure that all planet candidates that orbit the same star stay together on one sie(either all of them go to training or all to testing) so we can prevent data leakage and force our model to learn the general transit phyics instead of simply memorizing the properties of a specific star

In [57]:
from sklearn.model_selection import StratifiedGroupKFold

# Initialize StratifiedGroupKFold to get a single deterministic 80/20 train/test split
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

train_idx, test_idx = next(sgkf.split(df, df["y"], groups=df["kepid"]))

# final train and test dfs
df_train = df.iloc[train_idx].reset_index(drop=True)
df_test = df.iloc[test_idx].reset_index(drop=True)

print(f"Train: {len(df_train)} rows, positive rate {df_train['y'].mean():.3f}")
print(f"Test : {len(df_test)} rows, positive rate {df_test['y'].mean():.3f}")

Train: 6068 rows, positive rate 0.362
Test : 1519 rows, positive rate 0.362


## 3. Preprocessing and Feature Engineering

Different ML model families have different data requirements. To accommodate both tree-based models and linear/neural network models, we construct two separate preprocessing pathways while preventing data leakage (all statistics are computed on the training set only):

1. **Automated Skewness & Log-Transform Detection:**
   - Astronomical datasets naturally have massive scales. Some features are often right-skewed (a small number of extremely large values with a dense cluster of small values).
   - For these features, we apply a safe logarithmic transformation (ln(1 + x) using `np.log1p`) to control extreme outliers

2. **Tree-Based Preprocessing Pathway (`Xtr_tree`, `Xte_tree`):**
   
   - We impute missing values with the median and enable `add_indicator=True` so it shows that this feature was missing because sometimes a feature being missing is an indicator
   - Since Trees split data based on thresholds, they don't require feature scaling

3. **Linear Models & MLP Pathway (`Xtr_lin`, `Xte_lin`):**
   - Logistic classifiers, and neural networks are sensitive to feature scales so they need to be standardized 
   - We pass the data through a pipeline that does median imputation (with missing indicators) followed by a "StandardScaler" to center features to a mean of 0 and a standard deviation of 1.

In [58]:
#Which features are skewed and strictly positive? (computed on TRAIN only)
X_tr = df_train[clean_features]
#Calculates skewness for each feature on training data only
skew = X_tr.skew()
#Find features that are strictly positive (all non-missing values > 0) 
strictly_pos = []
for col in clean_features:
  if (X_tr[col].dropna() > 0).all():
    strictly_pos.append(col)

#we select to do log transform only on features that are strictly positive and skew>2
log_cols = []
for col in strictly_pos:
  if skew[col] > 2:
    log_cols.append(col)
    
print(f"{len(log_cols)} features selected for log transform:")
print(log_cols)

30 features selected for log transform:
['koi_period', 'koi_period_err1', 'koi_time0bk', 'koi_time0bk_err1', 'koi_duration', 'koi_depth', 'koi_ror', 'koi_srho', 'koi_prad', 'koi_sma', 'koi_teq', 'koi_insol', 'koi_dor', 'koi_max_sngle_ev', 'koi_max_mult_ev', 'koi_count', 'koi_tce_plnt_num', 'koi_srad', 'koi_fwm_sra_err', 'koi_fwm_sdec_err', 'koi_fwm_srao_err', 'koi_fwm_sdeco_err', 'koi_fwm_prao_err', 'koi_fwm_pdeco_err', 'koi_dicco_mra_err', 'koi_dicco_mdec_err', 'koi_dicco_msky_err', 'koi_dikco_mra_err', 'koi_dikco_mdec_err', 'koi_dikco_msky_err']


In [59]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X_train_raw = df_train[clean_features].copy()
X_test_raw = df_test[clean_features].copy()

#Apply Log Transforms directly using Pandas 
for c in log_cols:
  X_train_raw[c] = np.log1p(X_train_raw[c].clip(lower=0))
  X_test_raw[c] = np.log1p(X_test_raw[c].clip(lower=0))

#For Trees: fill missing values with the median (plus a missing indicator)
tree_preprocessor = SimpleImputer(strategy="median", add_indicator=True)

Xtr_tree = tree_preprocessor.fit_transform(X_train_raw)
Xte_tree = tree_preprocessor.transform(X_test_raw)

#For Linear Models / MLPs: impute missing values AND standardize scale
linear_pipeline = make_pipeline(
    SimpleImputer(strategy="median", add_indicator=True), StandardScaler()
)

Xtr_lin = linear_pipeline.fit_transform(X_train_raw)
Xte_lin = linear_pipeline.transform(X_test_raw)


print("Tree features   :", Xtr_tree.shape, Xte_tree.shape)
print("Linear/MLP feats:", Xtr_lin.shape, Xte_lin.shape)

Tree features   : (6068, 177) (1519, 177)
Linear/MLP feats: (6068, 177) (1519, 177)
